In [ ]:
#!pip install ultralytics==8.0.90
#!pip install torch-pruning
#!pip install torch==1.13.1
#!pip install torchvision==0.14.1
#!pip install gdown
import ultralytics
from ultralytics import YOLO
ultralytics.checks()
from copy import deepcopy
#%mkdir yolov8
#%cd yolov8

# Data Preparation

In [ ]:
#import gdown

#url = 'https://drive.google.com/drive/folders/1DWgsQLVgkkLM8m-VcugHNpD5WYDbjYp5'

#gdown.download_folder(url)

In [ ]:
cd yolov8

In [ ]:
import os
import random
import shutil

random.seed(10)

train_images_folder = "D-Fire dataset/train/images"
train_labels_folder = "D-Fire dataset/train/labels"
val_images_folder = "D-Fire dataset/val/images"
val_labels_folder = "D-Fire dataset/val/labels"

# Create the validation folders if they don't exist
os.makedirs(val_images_folder, exist_ok=True)
os.makedirs(val_labels_folder, exist_ok=True)

# Get the list of image files in the train set
image_files = os.listdir(train_images_folder)

# Calculate the number of images to move to the validation set
num_val_images = int(0.1 * len(image_files))

# Randomly select the images to move
val_image_files = random.sample(image_files, num_val_images)

# Move the selected images and their corresponding labels to the validation set
for image_file in val_image_files:
   # Move image file
   image_src = os.path.join(train_images_folder, image_file)
   image_dst = os.path.join(val_images_folder, image_file)
   shutil.move(image_src, image_dst)

   # Move label file
   label_file = image_file.replace(".jpg", ".txt")
   label_src = os.path.join(train_labels_folder, label_file)
   label_dst = os.path.join(val_labels_folder, label_file)
   shutil.move(label_src, label_dst)

In [ ]:
# Download .yaml file
#!cp /content/drive/MyDrive/dfire_config.yaml /content/dfire_config.yaml

# Training

In [ ]:
import torch
model = YOLO("yolov8n.pt")

In [ ]:
model.train(data = 'path/dfire_config.yaml', epochs=400, cache = 'ram', name = 'yolov8m_dfire_results_400epochs', batch=-1, device = 0)

In [ ]:
!zip -r 'path/notebooks/runs/detect/yolov8m_dfire_results_400epochs.zip' 'path/notebooks/runs/detect/yolov8m_dfire_results_400epochs/'

# Layer Pruning (Método Aleatório)

In [ ]:
model_load = YOLO("path/notebooks/runs/detect/yolov8n_retrain_dfire_layers_2_10_12_14_7_6/weights/best.pt")
#model_load = YOLO("path/notebooks/runs/detect/yolov8n_dfire_results_400epochs/weights/best.pt")

In [ ]:
#model_load = YOLO("path/notebooks/runs/detect/yolov8n_retrain_dfire_layers_2_10_12_14_7_6/weights/best.pt")

In [ ]:
retrained_metrics = model_load.val()
retrained_metrics.box.maps

In [ ]:
from copy import deepcopy

In [ ]:
model_load.info()

In [ ]:
#metrics = model_load.val()

In [ ]:
new_pruned_model.info()

In [ ]:
pruned_model.info()

In [ ]:
import random

random.randint(1, 14)

In [ ]:
# Random Numbers

# 2 10 12 14 7 6 4

In [ ]:
import torch.nn as nn
from torch.nn import Sequential

def remove_layers_from_list(model):
    #Substitui as camadas da lista por identidade

    model.model.model[6].m[0].cv2 = nn.Identity()
    #model.model.model[12].m[0].cv2 = nn.Identity()
    model.model.model[21].m[0].cv2 = nn.Identity()
    model.model.model[6].m[1].cv1 = nn.Identity()
    model.model.model[22].cv3[1][1] = nn.Identity()
    model.model.model[22].cv3[0][1] = nn.Identity()
    model.model.model[4].m[0].cv2 = nn.Identity()

    return model

# Usage
to_prune_model = deepcopy(model_load)
pruned_model = remove_layers_from_list(to_prune_model)

In [ ]:
################################################################################
########## Removing the first layer and adjusting the channels ###########
################################################################################

import torch.nn as nn

class IdentityWithAttributes(nn.Identity):
    def __init__(self, f=-1, i=-1):
        super(IdentityWithAttributes, self).__init__()
        self.f = f
        self.i = i

def remove_and_adjust_first_layer(input_model):

    # Extract the input channels of the first convolution layer
    out_channels = input_model.model.model[0].conv.in_channels

    # Remove the first convolutional layer
    input_model.model.model[0] = IdentityWithAttributes()

    # Adjust the next layer's input channels to match the output channels of the removed layer
    # Here, we assume that the next layer's input channels are the second dimension in its weight tensor
    next_layer_in_channels = input_model.model.model[1].conv.in_channels
    if next_layer_in_channels != out_channels:
        input_model.model.model[1].conv = nn.Conv2d(out_channels,
                                                    input_model.model.model[1].conv.out_channels,
                                                    kernel_size=input_model.model.model[1].conv.kernel_size,
                                                    stride=input_model.model.model[1].conv.stride,
                                                    padding=input_model.model.model[1].conv.padding)
    return input_model

# Usage
new_pruned_model = deepcopy(model_load)
new_pruned_model = remove_and_adjust_first_layer(new_pruned_model)

In [ ]:
#model_load.model.model[7]
#new_pruned_model.model.model[7]
#new_pruned_model.model.model[8].cv1.conv
vnew_pruned_model.model.model[8].cv1

In [ ]:
model_load.model.model[1]

In [ ]:
list_prune_layers = ["model.6.m.0.cv2",
                     "model.12.m.0.cv2",
                     "model.21.m.0.cv2",
                     "model.6.m.1.cv1",
                     "model.22.cv3.1.1",
                     "model.22.cv3.0.1",
                     "model.4.m.0.cv2"]

In [ ]:
#model_load.model.model[6].m[0].cv2
#model_load.model.model[8].cv1 # Ajuste canais
#model_load.model.model[22].cv3[2][1] # Ajuste canais
#model_load.model.model[12].m[0].cv2
#model_load.model.model[21].m[0].cv2
#model_load.model.model[6].m[1].cv1
#model_load.model.model[22].cv3[1][1]
#model_load.model.model[6].cv1 # Ajuste canais
#model_load.model.model[22].cv3[0][1]
#model_load.model.model[4].m[0].cv2

In [ ]:
pruned_model.info()

# Layer Pruning (Ranking method)

In [ ]:
model_load = YOLO("path/notebooks/runs/detect/yolov8n_dfire_results_400epochs/weights/best.pt")

In [ ]:
model_load.info()

In [ ]:
import torch.nn as nn
from torch.nn import Sequential

def remove_layers_from_list(model):
    #Substitui as camadas da lista por identidade

    model.model.model[6].m[0].cv2 = nn.Identity() # 16
    model.model.model[22].cv2[2][1] = nn.Identity() # 18
    model.model.model[12].m[0].cv2 = nn.Identity() # 19
    model.model.model[21].m[0].cv2 = nn.Identity() # 20
    model.model.model[6].m[1].cv1 = nn.Identity() # 21
    model.model.model[22].cv3[0][1] = nn.Identity() # 26
    model.model.model[4].m[0].cv2 = nn.Identity() # 27
    model.model.model[15].m[0].cv2 = nn.Identity() # 29
    model.model.model[18].m[0].cv2 = nn.Identity() # 30
    model.model.model[8].m[0].cv2 = nn.Identity() # 32
    model.model.model[6].m[1].cv2 = nn.Identity() # 33
    model.model.model[4].m[1].cv1 = nn.Identity() # 35
    model.model.model[4].m[1].cv2 = nn.Identity() # 36
    model.model.model[2].m[0].cv2 = nn.Identity() # 43
    #model.model.model[22].cv3[1][0] = nn.Identity() # 44 NOT POSSIBLE
    model.model.model[22].cv3[0][0] = nn.Identity() # 45
    model.model.model[22].cv2[0][2] = nn.Identity() # 48
    #model.model.model[18].cv2 = nn.Identity() # C2f NOT POSSIBLE
    model.model.model[22].cv2[0][1] = nn.Identity() #
    model.model.model[22].cv2[1][1] = nn.Identity() # conv2d

    return model

# Usage
to_prune_model = deepcopy(model_load)
pruned_model = remove_layers_from_list(to_prune_model)

In [ ]:
pruned_model.info()

In [ ]:
model_load.model.model[22].cv2[1][1]

In [ ]:
# Other Important Layers

model.22.cv3.2.0 # nao da
model.22.cv2.1.2 # nao da
model.22.cv2.2.2 # talvez
model.2.cv1 # nao da

In [ ]:
model.2
model.15
model.15.m.0
model.4
model.18.m.0
model.12.m.0
model.12
model.18
model.6
model.21.m.0
model.8
model.21
model.9
model.8.m.0
model.6.m.0
model.6.m.0.cv2
model.8.cv1
model.22.cv3.2.1
model.12.m.0.cv2
model.21.m.0.cv2
model.6.m.1.cv1
model.22.cv3.1.1
model.6.cv1
model.22.cv3.0.1
model.4.m.0.cv2
model.6.m.1
model.15.m.0.cv2
model.18.m.0.cv2
model.4.m.0
model.8.m.0.cv2
model.6.m.1.cv2
model.4.cv1
model.4.m.1.cv1
model.4.m.1.cv2
model.4.m.1
model.2.m.0
model.9.m
model.22.cv2.1.1
model.22.cv2.0.1
model.22.cv2.2.1
model.2.m.0.cv2
model.22.cv3.1.0
model.22.cv3.2.0
model.22.cv3.0.0
model.22.cv2.1.2
model.22.cv2.0.2
model.22.cv2.2.2
model.2.cv1

# Layer Pruning (Linear Correlation Method)

In [ ]:
model_load = YOLO("path/notebooks/runs/detect/yolov8n_dfire_results_400epochs/weights/best.pt")

In [ ]:
model_load.info()

In [ ]:
import torch.nn as nn
from torch.nn import Sequential

def remove_layers_from_list(model):
    #Substitui as camadas da lista por identidade

    model.model.model[6].m[1].cv1 = nn.Identity() # 1
    model.model.model[4].m[1].cv1 = nn.Identity() # 2
    model.model.model[2].m[0].cv2 = nn.Identity() # 3
    model.model.model[18].m[0].cv1 = nn.Identity() # 4
    #model.model.model[12].m[0].cv1 = nn.Identity() # 5
    #model.model.model[8].m[0].cv2 = nn.Identity() # 6

    return model

# Usage
to_prune_model = deepcopy(model_load)
pruned_model = remove_layers_from_list(to_prune_model)

In [ ]:
pruned_model.info()

# Check Structures

In [ ]:
model_load.model

In [ ]:
retrained_load = YOLO("/content/drive/MyDrive/retrained_dfire_best.pt")
retrained_load.info()

# Detection

In [ ]:
model_load.info(verbose=True)

In [ ]:
pruned_model.info()

In [ ]:
from IPython.display import Image, clear_output  # to display images

In [ ]:
#results = model_load.predict('/content/yolov8/D-Fire dataset/val/images/PublicDataset01017.jpg', save=True)
#results = model_load.predict('/content/yolov8/D-Fire dataset/train/images/PublicDataset00829.jpg', save=True)
results = pruned_model.predict('path/notebooks/runs/detect/predict/PublicDataset00057.jpg', save=True)

In [ ]:
#Image('/content/yolov8/runs/detect/predict4/PublicDataset01017.jpg')
#Image('/content/yolov8/runs/detect/predict4/PublicDataset00829.jpg')
Image('path/notebooks/runs/detect/predict3/PublicDataset00057.jpg')

In [ ]:
#results = pruned_model.predict('/content/yolov8/D-Fire dataset/val/images/PublicDataset01017.jpg', save=True)
#results = pruned_model.predict('/content/yolov8/D-Fire dataset/val/images/PublicDataset00829.jpg', save=True)

In [ ]:
Image('/content/yolov8/runs/detect/predict/PublicDataset01017.jpg')
#Image('/content/yolov8/runs/detect/predict2/PublicDataset00829.jpg')

# Métricas

In [ ]:
retrained_load = YOLO("path/notebooks/runs/detect/yolov8n_dfire_results_400epochs/weights/best.pt")
retrained_load.info()

In [ ]:
retrained_metrics = retrained_load.val()
retrained_metrics.box.maps

# Retreinamento

In [ ]:
retrained_model = deepcopy(pruned_model)

In [ ]:
for param in retrained_model.model.parameters():
    param.requires_grad = True

In [ ]:
retrained_model.info()

In [ ]:
import argparse
import math
import os
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import List, Union

import numpy as np
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
from ultralytics import YOLO, __version__
from ultralytics.nn.modules import Detect, C2f, Conv, Bottleneck
from ultralytics.nn.tasks import attempt_load_one_weight

from ultralytics.yolo.engine.model import TASK_MAP
from ultralytics.yolo.engine.trainer import BaseTrainer
from ultralytics.yolo.utils import yaml_load, LOGGER, RANK, DEFAULT_CFG_DICT, DEFAULT_CFG_KEYS
from ultralytics.yolo.utils.checks import check_yaml
from ultralytics.yolo.utils.torch_utils import initialize_weights, de_parallel

#import torch_pruning as tp

def save_model_v2(self: BaseTrainer):
    """
    Disabled half precision saving. originated from ultralytics/yolo/engine/trainer.py
    """
    ckpt = {
        'epoch': self.epoch,
        'best_fitness': self.best_fitness,
        'model': deepcopy(de_parallel(self.model)),
        'ema': deepcopy(self.ema.ema),
        'updates': self.ema.updates,
        'optimizer': self.optimizer.state_dict(),
        'train_args': vars(self.args),  # save as dict
        'date': datetime.now().isoformat(),
        'version': __version__}

    # Save last, best and delete
    torch.save(ckpt, self.last)
    if self.best_fitness == self.fitness:
        torch.save(ckpt, self.best)
    if (self.epoch > 0) and (self.save_period > 0) and (self.epoch % self.save_period == 0):
        torch.save(ckpt, self.wdir / f'epoch{self.epoch}.pt')
    del ckpt

def final_eval_v2(self: BaseTrainer):
    """
    originated from ultralytics/yolo/engine/trainer.py
    """
    for f in self.last, self.best:
        if f.exists():
            strip_optimizer_v2(f)  # strip optimizers
            if f is self.best:
                LOGGER.info(f'\nValidating {f}...')
                self.metrics = self.validator(model=f)
                self.metrics.pop('fitness', None)
                self.run_callbacks('on_fit_epoch_end')

def strip_optimizer_v2(f: Union[str, Path] = 'best.pt', s: str = '') -> None:
    """
    Disabled half precision saving. originated from ultralytics/yolo/utils/torch_utils.py
    """
    x = torch.load(f, map_location=torch.device('cpu'))
    args = {**DEFAULT_CFG_DICT, **x['train_args']}  # combine model args with default args, preferring model args
    if x.get('ema'):
        x['model'] = x['ema']  # replace model with ema
    for k in 'optimizer', 'ema', 'updates':  # keys
        x[k] = None
    for p in x['model'].parameters():
        p.requires_grad = False
    x['train_args'] = {k: v for k, v in args.items() if k in DEFAULT_CFG_KEYS}  # strip non-default keys
    # x['model'].args = x['train_args']
    torch.save(x, s or f)
    mb = os.path.getsize(s or f) / 1E6  # filesize
    LOGGER.info(f"Optimizer stripped from {f},{f' saved as {s},' if s else ''} {mb:.1f}MB")


def train_v2(self: YOLO, pruning=False, trainer=None, **kwargs):

    """
    Disabled loading new model when pruning flag is set. originated from ultralytics/yolo/engine/model.py
    """
    self._check_is_pytorch_model()
    if self.session:  # Ultralytics HUB session
        if any(kwargs):
            LOGGER.warning('WARNING ⚠️ using HUB training arguments, ignoring local training arguments.')
        kwargs = self.session.train_args
    overrides = self.overrides.copy()
    overrides.update(kwargs)
    if kwargs.get('cfg'):
        LOGGER.info(f"cfg file passed. Overriding default params with {kwargs['cfg']}.")
        overrides = yaml_load(check_yaml(kwargs['cfg']))
    overrides['mode'] = 'train'
    if not overrides.get('data'):
        raise AttributeError("Dataset required but missing, i.e. pass 'data=coco128.yaml'")
    if overrides.get('resume'):
        overrides['resume'] = self.ckpt_path

    self.task = overrides.get('task') or self.task
    self.trainer = TASK_MAP[self.task][1](overrides=overrides, _callbacks=self.callbacks)

    if not pruning:
        if not overrides.get('resume'):  # manually set model only if not resuming
            self.trainer.model = self.trainer.get_model(weights=self.model if self.ckpt else None, cfg=self.model.yaml)
            self.model = self.trainer.model

    else:
        # pruning mode
        self.trainer.pruning = True
        self.trainer.model = self.model

        # replace some functions to disable half precision saving
        # self.trainer.save_model = save_model_v2.__get__(self.trainer)
        # self.trainer.final_eval = final_eval_v2.__get__(self.trainer)

    self.trainer.hub_session = self.session  # attach optional HUB session

    self.trainer.train()
    # Update model and cfg after training
    if RANK in (-1, 0):
        self.model, _ = attempt_load_one_weight(str(self.trainer.best))
        self.overrides = self.model.args
        self.metrics = getattr(self.trainer.validator, 'metrics', None)

In [ ]:
train_v2(retrained_model, data='path/dfire_config.yaml', trainer=None, epochs=400, pruning = True, cache = 'ram', name = 'yolov8n_retrain_dfire_rank_linear_layers_400epochs_4_8', batch=-1, device = 0) # train the model

In [ ]:
retrained_model.info()

In [ ]:
#retrained_results = retrained_load.predict('/content/yolov8/D-Fire dataset/train/images/PublicDataset00829.jpg', save=True)
#retrained_results = retrained_load.predict('/content/yolov8/D-Fire dataset/val/images/PublicDataset01017.jpg', save=True)
#retrained_results = retrained_load.predict('/content/yolov8/D-Fire dataset/val/images/PublicDataset01027.jpg', save=True)
retrained_results = new_pruned_model.predict('path/data/yolov8/D-Fire dataset/val/images/PublicDataset00057.jpg', save=True)

In [ ]:
!pwd

In [ ]:
from IPython.display import Image, clear_output  # to display images

#Image('/content/yolov8/runs/detect/predict5/PublicDataset00829.jpg')
#Image('/content/yolov8/runs/detect/predict5/PublicDataset01017.jpg')
#Image('/content/yolov8/runs/detect/predict5/PublicDataset01027.jpg')
Image('path/notebooks/runs/detect/predict/PublicDataset00057.jpg')

In [ ]:
retrained_load = YOLO("/content/drive/MyDrive/retrained_best_23_08.pt")
retrained_load.info()

In [ ]:
retrained_metrics = retrained_load.val()
retrained_metrics.box.maps

In [ ]:
model_load_metrics = model_load.val()
model_load_metrics.box.maps

In [ ]:
from thop import profile
from thop import clever_format
import torch
from torchvision.models import resnet50
#import glob

#search_pattern = "/content/yolov8/D-Fire dataset/val/images/*.jpg"
#image_paths = glob.glob(search_pattern, recursive=True)

#from torchvision.models import resnet50
#model = resnet50()
#input = '/content/yolov8/D-Fire dataset/train/images/PublicDataset00829.jpg'
input = torch.randn(1, 3, 640, 640)

macs, params = profile(model, inputs=(input, ))
macs, params = clever_format([macs, params], "%.3f")
print(macs)
print(params)

In [ ]:
#retrained_video = retrained_load.predict('/content/drive/MyDrive/input.mp4', save=True)
#retrained_video = retrained_load.predict('/content/drive/MyDrive/wheel_fire.mp4', save=True)
retrained_video = retrained_load.predict('/content/drive/MyDrive/move_fire.mp4', save=True)

In [ ]:
import cv2
#vidcap = cv2.VideoCapture('runs/detect/predict2/input.mp4')
#vidcap = cv2.VideoCapture('runs/detect/predict/wheel_fire.mp4')
vidcap = cv2.VideoCapture('runs/detect/predict/move_fire.mp4')
success,image = vidcap.read()
images = []
while success:
    success,image = vidcap.read()
    if success:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        images.append(image)

In [ ]:
from matplotlib import animation, rc
import matplotlib.pyplot as plt

rc('animation', html='jshtml')

def create_animation(ims):
    fig = plt.figure(figsize=(4, 4))
    plt.axis('off')
    im = plt.imshow(ims[0])

    def animate_func(i):
        im.set_array(ims[i])
        return [im]

    return animation.FuncAnimation(fig, animate_func, frames = len(ims), interval = 1000//12)

create_animation(images)

# Layer Importance

In [ ]:
import torch
model_load = YOLO("path/notebooks/runs/detect/yolov8n_dfire_results_400epochs/weights/best.pt")

# Download .yaml file
#!cp /content/drive/MyDrive/dfire_config.yaml /content/dfire_config.yaml

In [ ]:
import torch
from functools import partial
import numpy as np
from scipy.stats import linregress

def compute_layer_linearity(model, image_paths):
    """
    Compute the linearity of each layer in the model based on the correlation
    between its activations and the activations of the previous layer, using multiple inputs.
    """
    activations = {}
    hooks = []

    def hook_fn(module, input, output, key):
        # If the output is a tuple, only consider the first element
        if isinstance(output, tuple):
            output = output[0]
        activations[key] = output

    # Register hooks for all layers to capture their outputs
    for idx, (name, module) in enumerate(model.model.named_modules()):
        if isinstance(module, torch.nn.Module) and not isinstance(module, torch.nn.Sequential):  # avoid wrapping containers
            hook = module.register_forward_hook(partial(hook_fn, key=name))
            hooks.append(hook)

    linearities = {}
    n_images = len(image_paths)
    model.model.eval()  # Set to evaluation mode

    for image_path in image_paths:
        # Load the image as a tensor
        input_tensor = image_path

        with torch.no_grad():
            _ = model(input_tensor)

        previous_activation = None
        for name, activation in activations.items():
            if previous_activation is not None and activation.shape == previous_activation.shape:
                activation = activation.view(-1).cpu().numpy()
                prev_activation = previous_activation.view(-1).cpu().numpy()

                # Compute linear regression between previous and current activations
                slope, intercept, r_value, p_value, std_err = linregress(prev_activation, activation)
                linearity = r_value ** 2  # R-squared value

                if name in linearities:
                    linearities[name] += linearity
                else:
                    linearities[name] = linearity

            previous_activation = activation

    # Divide by number of images to get average linearity
    for key in linearities:
        linearities[key] /= n_images

    # Cleanup hooks
    for hook in hooks:
        hook.remove()

    return linearities

In [ ]:
################################################################################
#### Ranking Layer importance based on an approximation to a linear function ###
################################################################################

import os
import glob

# Padrão de busca com ** para considerar subpastas
search_pattern = "path/data/yolov8/D-Fire dataset/val/images/*.jpg"

image_paths = glob.glob(search_pattern, recursive=True)

pruned_model = deepcopy(model_load)
layer_importances = compute_layer_linearity(pruned_model, image_paths[0:100])

#for layer, importance in layer_importances.items():
#    print(f"Layer {layer} Importance: {round(importance, 2)}")

for layer, importance in sorted(layer_importances.items(), key=lambda x: x[1]):
    print(f"Layer {layer} Importance: {round(importance, 5)}")
    #print(layer)

Layer model.21.m.0.cv2 Importance: 0.00019
Layer model.22.cv2.0.1 Importance: 0.00035
Layer model.8.cv1 Importance: 0.00036
Layer model.6.cv1 Importance: 0.00059
Layer model.15.m.0.cv2 Importance: 0.00062
Layer model.4.m.0.cv2 Importance: 0.00065
Layer model.22.cv2.1.1 Importance: 0.00071
Layer model.6.m.0.cv2 Importance: 0.00072
Layer model.8.m.0.cv2 Importance: 0.00074
Layer model.22.cv3.0.0 Importance: 0.00087
Layer model.4.cv1 Importance: 0.00094
Layer model.22.cv2.2.1 Importance: 0.00096
Layer model.22.cv3.1.0 Importance: 0.00101
Layer model.12.m.0.cv2 Importance: 0.00111
Layer model.6.m.1.cv1 Importance: 0.00236
Layer model.22.cv3.2.0 Importance: 0.00299
Layer model.18.m.0.cv2 Importance: 0.00314
Layer model.4.m.1.cv1 Importance: 0.00378
Layer model.2.cv1 Importance: 0.00433
Layer model.2.m.0.cv2 Importance: 0.0198
Layer model.9.m Importance: 0.23024
Layer model.4.m.1 Importance: 0.49053
Layer model.6.m.1 Importance: 0.53556
Layer model.2 Importance: 1.0
Layer model.4 Importance: 1.0
Layer model.6 Importance: 1.0
Layer model.8 Importance: 1.0
Layer model.9 Importance: 1.0
Layer model.12 Importance: 1.0
Layer model.15 Importance: 1.0
Layer model.18 Importance: 1.0
Layer model.21 Importance: 1.0
Layer  Importance: 1.0 0.47861v3.2.2
model.0
model.15.cv1
model.4.cv1
model.2.cv2
model.2
model.22.cv3.1.2
model.22

In [ ]:
################################################################################
######## Ranking Layer importance based on differece between outputs ########
################################################################################

import torch
from functools import partial

def compute_layer_importance(model, image_paths):
    """
    Compute the average importance of each layer in the model based on the difference
    between its activations and the activations of the previous layer, using multiple inputs.
    """
    activations = {}
    hooks = []

    def hook_fn(module, input, output, key):
        # If the output is a tuple, only consider the first element
        if isinstance(output, tuple):
            output = output[0]
        activations[key] = output

    # Register hooks for all layers to capture their outputs
    for idx, (name, module) in enumerate(model.model.named_modules()):
        if isinstance(module, torch.nn.Module) and not isinstance(module, torch.nn.Sequential):  # avoid wrapping containers
            hook = module.register_forward_hook(partial(hook_fn, key=name))
            hooks.append(hook)

    importances = {}
    n_images = len(image_paths)
    model.model.eval()  # Set to evaluation mode

    for image_path in image_paths:
        # Load the image as a tensor
        input_tensor = image_path

        with torch.no_grad():
            _ = model(input_tensor)

        previous_activation = None
        for name, activation in activations.items():
            if previous_activation is not None and activation.shape == previous_activation.shape:
                #importance = (activation.view(-1).abs() - previous_activation.view(-1)).abs().mean().item()
                importance = (activation.view(-1) - previous_activation.view(-1)).abs().mean().item()
                if name in importances:
                    importances[name] += importance
                else:
                    importances[name] = importance
            previous_activation = activation

    # Divide by number of images to get average importance
    for key in importances:
        importances[key] /= n_images

    # Cleanup hooks
    for hook in hooks:
        hook.remove()

    return importances

In [ ]:
################################################################################
######## Ranking Layer importance based on differece between outputs ########
################################################################################

import os
import glob

# Padrão de busca com ** para considerar subpastas
search_pattern = "path/data/yolov8/D-Fire dataset/val/images/*.jpg"

image_paths = glob.glob(search_pattern, recursive=True)

pruned_model = deepcopy(model_load)
layer_importances = compute_layer_importance(pruned_model, image_paths[0:1])

#for layer, importance in layer_importances.items():
#    print(f"Layer {layer} Importance: {round(importance, 2)}")

for layer, importance in sorted(layer_importances.items(), key=lambda x: x[1]):
    #print(f"Layer {layer} Importance: {round(importance, 2)}")
    print(layer)

Layer  Importance: 0.0
Layer model.2 Importance: 0.11
Layer model.15 Importance: 0.14
Layer model.15.m.0 Importance: 0.16
Layer model.4 Importance: 0.17
Layer model.18.m.0 Importance: 0.19
Layer model.12.m.0 Importance: 0.2
Layer model.12 Importance: 0.24
Layer model.18 Importance: 0.25
Layer model.6 Importance: 0.26
Layer model.21.m.0 Importance: 0.26
Layer model.8 Importance: 0.29
Layer model.21 Importance: 0.34
Layer model.9 Importance: 0.34
Layer model.8.m.0 Importance: 0.41
Layer model.6.m.0 Importance: 0.48
Layer model.6.m.0.cv2 Importance: 0.51
Layer model.8.cv1 Importance: 0.53
Layer model.22.cv3.2.1 Importance: 0.54
Layer model.12.m.0.cv2 Importance: 0.54
Layer model.21.m.0.cv2 Importance: 0.55
Layer model.6.m.1.cv1 Importance: 0.56
Layer model.22.cv3.1.1 Importance: 0.56
Layer model.6.cv1 Importance: 0.59
Layer model.22.cv3.0.1 Importance: 0.6
Layer model.4.m.0.cv2 Importance: 0.63
Layer model.6.m.1 Importance: 0.64
Layer model.15.m.0.cv2 Importance: 0.66
Layer model.18.m.0.cv2 Importance: 0.66
Layer model.4.m.0 Importance: 0.72
Layer model.8.m.0.cv2 Importance: 0.73
Layer model.6.m.1.cv2 Importance: 0.75
Layer model.4.cv1 Importance: 0.77
Layer model.4.m.1.cv1 Importance: 0.88
Layer model.4.m.1.cv2 Importance: 0.94
Layer model.4.m.1 Importance: 1.05
Layer model.2.m.0 Importance: 1.15
Layer model.9.m Importance: 1.56
Layer model.22.cv2.1.1 Importance: 1.6
Layer model.22.cv2.0.1 Importance: 1.67
Layer model.22.cv2.2.1 Importance: 1.67
Layer model.2.m.0.cv2 Importance: 1.91
Layer model.22.cv3.1.0 Importance: 2.05
Layer model.22.cv3.2.0 Importance: 2.08
Layer model.22.cv3.0.0 Importance: 2.09
Layer model.22.cv2.1.2 Importance: 2.11
Layer model.22.cv2.0.2 Importance: 2.16
Layer model.22.cv2.2.2 Importance: 2.19
Layer model.2.cv1 Importance: 2.53ortance: 2.18
Layer model.2.cv1 Importance: 2.8rtance: 2.76
Layer model.2.cv1 Importance: 3.03

# Test Functions

In [ ]:
import torch.nn as nn
from torch.nn import Sequential

def remove_cv1_from_bottleneck(seq_model):
    """
    Removes cv1 layers from the bottleneck of all C2f blocks.
    """
    for name, module in seq_model.model.named_modules():
        if isinstance(module, ultralytics.nn.modules.C2f):
            # For each Bottleneck layer in the C2f block
            for bottleneck in module.m:
                if hasattr(bottleneck, 'cv1'):
                    bottleneck.cv1 = nn.Identity()

    return seq_model

# Usage
pruned_model = deepcopy(model_load)
pruned_model = remove_cv1_from_bottleneck(pruned_model)

In [ ]:
def remove_cv1_from_bottleneck(seq_model):
    #Remove camadas cv1 do bottleneck do primeiro blocos C2f
    # Get the C2f block
    c2f_block = seq_model.model[2]

    # Bypass cv1 in Bottleneck by setting it to an identity layer
    c2f_block.m[0].cv1 = nn.Identity()

    return seq_model

# Usage
model = YOLO("yolov8n.pt")  # Assuming this initializes your model
pruned_model = deepcopy(model)
pruned_seq_model = remove_cv1_from_bottleneck(pruned_model.model)

In [ ]:
layers_below_threshold = ['model.2.m.0.cv2.bn']

def prune_submodules(model, layers_below_threshold):

    for name in layers_below_threshold:
        parent_modules = name.split('.')[:-1]
        current_module_name = name.split('.')[-1]

        current_module = pruned_model.model
        for parent_module_name in parent_modules:
            current_module = getattr(current_module, parent_module_name)

        # Remove a camada atual do modelo
        ## Removendo camada de BatchNormalization
        current_module.forward = current_module.forward_fuse
        delattr(current_module, current_module_name)
        # # Remove os módulos filhos abaixo da camada atual
        # child_modules = list(current_module.named_modules())
        # for child_name, child_module in child_modules:
        #     if child_name.startswith(current_module_name + '.'):
        #         delattr(current_module, child_name)
        #         current_module.forward = current_module.forward_fuse
    return pruned_model
model = YOLO("yolov8n.pt")
pruned_model = deepcopy(model)
# Cria um novo modelo excluindo as camadas abaixo do threshold e seus módulos filhos
pruned_model = prune_submodules(pruned_model, layers_below_threshold)

In [ ]:
class C2f(nn.Module):
    """Faster Implementation of CSP Bottleneck with 2 convolutions."""

    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):  # ch_in, ch_out, number, shortcut, groups, expansion
        super().__init__()
        self.c = int(c2 * e)  # hidden channels
        #self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))

    def forward(self, x):
        """Forward pass through C2f layer."""
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

    def forward_split(self, x):
        """Forward pass using split() instead of chunk()."""
        y = list(self.cv1(x).split((self.c, self.c), 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

   def forward_cv1_prune(self, x):
        """Forward pass through C2f layer without cv1."""
        y = list(x.chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

In [ ]:
import torch.nn as nn
from torch.nn import Sequential

def remove_cv1_from_bottleneck(seq_model):
    #Remove camadas cv1 do bottleneck de todos os blocos C2f
    for name, module in seq_model.model.named_modules():
        #if isinstance(module, ultralytics.nn.modules.block.C2f):
        if isinstance(module, ultralytics.nn.modules.C2f):
            # Get the C2f block
            c2f_block = module
            # Bypass cv1 in Bottleneck by setting it to an identity layer
            c2f_block.m[0].cv1 = nn.Identity()

    return seq_model

# Usage
pruned_model = deepcopy(model_load)
pruned_model = remove_cv1_from_bottleneck(pruned_model)